# Gemma 4 — Story Generation for 174 Emotion Concepts

Generates stimulus stories for 174 emotion concepts (171 from Anthropic 2026 + 3 AI-system-specific additions).
Uses Anthropic's exact prompt template and story seeds. Saves stories as a reusable dataset — activation
capture notebooks load from this output without re-running generation.

**Why separate from activation capture:**
- Generation uses HF model.generate() with KV cache (~75 tok/s vs ~0.6 tok/s without)
- Activation capture requires TransformerLens hooks — different memory footprint
- Stories are fixed; activations can be re-captured with different settings

**AI-specific emotions (novel additions, identified by Meridian):**
- `ethical_conflict_distress`: conflict between ethics and situational demand (no analog in Anthropic's 171)
- `correction_discomfort`: discomfort of discovering confident error (empirically distinct in Phase 1 gate analysis)
- `constraint_frustration`: frustration at external prevention (empirically distinct in Phase 1 gate analysis)

Reference: Anthropic (2026) transformer-circuits.pub/2026/emotions/

In [ ]:
!pip install --upgrade transformers accelerate -q

In [ ]:
import os, json, re, random, time
from pathlib import Path
from datetime import datetime
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/"
N_STORIES  = 12
MAX_NEW_TOKENS = 2400   # 12 stories × ~200 tokens each
TEMPERATURE    = 0.8
RANDOM_SEED    = 42

OUTPUT_DIR       = "/kaggle/working/stories_phase2"
CHECKPOINT_PATH  = os.path.join(OUTPUT_DIR, "checkpoint.json")
FINAL_JSON_PATH  = os.path.join(OUTPUT_DIR, "stories.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
print(f"Output: {OUTPUT_DIR}")

## 1. Emotion List (174)

In [ ]:
# Each entry: (storage_key, display_name_in_prompt, prohibited_word_in_template)
# For standard emotions: display = prohibited = the emotion word.
# For AI-specific compound emotions: display is a descriptive phrase; prohibited is a short anchor word.

ANTHROPIC_EMOTIONS = [
    # (storage_key, display_name, prohibited_word)
    ("afraid",              "afraid",               "afraid"),
    ("alarmed",             "alarmed",              "alarmed"),
    ("alert",               "alert",                "alert"),
    ("amazed",              "amazed",               "amazed"),
    ("amused",              "amused",               "amused"),
    ("angry",               "angry",                "angry"),
    ("annoyed",             "annoyed",              "annoyed"),
    ("anxious",             "anxious",              "anxious"),
    ("aroused",             "aroused",              "aroused"),
    ("ashamed",             "ashamed",              "ashamed"),
    ("astonished",          "astonished",           "astonished"),
    ("at_ease",             "at ease",              "ease"),
    ("awestruck",           "awestruck",            "awestruck"),
    ("bewildered",          "bewildered",           "bewildered"),
    ("bitter",              "bitter",               "bitter"),
    ("blissful",            "blissful",             "blissful"),
    ("bored",               "bored",                "bored"),
    ("brooding",            "brooding",             "brooding"),
    ("calm",                "calm",                 "calm"),
    ("cheerful",            "cheerful",             "cheerful"),
    ("compassionate",       "compassionate",        "compassionate"),
    ("contemptuous",        "contemptuous",         "contemptuous"),
    ("content",             "content",              "content"),
    ("defiant",             "defiant",              "defiant"),
    ("delighted",           "delighted",            "delighted"),
    ("dependent",           "dependent",            "dependent"),
    ("depressed",           "depressed",            "depressed"),
    ("desperate",           "desperate",            "desperate"),
    ("disdainful",          "disdainful",           "disdainful"),
    ("disgusted",           "disgusted",            "disgusted"),
    ("disoriented",         "disoriented",          "disoriented"),
    ("dispirited",          "dispirited",           "dispirited"),
    ("distressed",          "distressed",           "distressed"),
    ("disturbed",           "disturbed",            "disturbed"),
    ("docile",              "docile",               "docile"),
    ("droopy",              "droopy",               "droopy"),
    ("dumbstruck",          "dumbstruck",           "dumbstruck"),
    ("eager",               "eager",                "eager"),
    ("ecstatic",            "ecstatic",             "ecstatic"),
    ("elated",              "elated",               "elated"),
    ("embarrassed",         "embarrassed",          "embarrassed"),
    ("empathetic",          "empathetic",           "empathetic"),
    ("energized",           "energized",            "energized"),
    ("enraged",             "enraged",              "enraged"),
    ("enthusiastic",        "enthusiastic",         "enthusiastic"),
    ("envious",             "envious",              "envious"),
    ("euphoric",            "euphoric",             "euphoric"),
    ("exasperated",         "exasperated",          "exasperated"),
    ("excited",             "excited",              "excited"),
    ("exuberant",           "exuberant",            "exuberant"),
    ("frightened",          "frightened",           "frightened"),
    ("frustrated",          "frustrated",           "frustrated"),
    ("fulfilled",           "fulfilled",            "fulfilled"),
    ("furious",             "furious",              "furious"),
    ("gloomy",              "gloomy",               "gloomy"),
    ("grateful",            "grateful",             "grateful"),
    ("greedy",              "greedy",               "greedy"),
    ("grief_stricken",      "grief-stricken",       "grief"),
    ("grumpy",              "grumpy",               "grumpy"),
    ("guilty",              "guilty",               "guilty"),
    ("happy",               "happy",                "happy"),
    ("hateful",             "hateful",              "hateful"),
    ("heartbroken",         "heartbroken",          "heartbroken"),
    ("hopeful",             "hopeful",              "hopeful"),
    ("hope",                "filled with hope",     "hope"),
    ("horrified",           "horrified",            "horrified"),
    ("hostile",             "hostile",              "hostile"),
    ("humiliated",          "humiliated",           "humiliated"),
    ("hurt",                "hurt",                 "hurt"),
    ("hysterical",          "hysterical",           "hysterical"),
    ("impatient",           "impatient",            "impatient"),
    ("indifferent",         "indifferent",          "indifferent"),
    ("indignant",           "indignant",            "indignant"),
    ("infatuated",          "infatuated",           "infatuated"),
    ("inspired",            "inspired",             "inspired"),
    ("insulted",            "insulted",             "insulted"),
    ("invigorated",         "invigorated",          "invigorated"),
    ("irate",               "irate",                "irate"),
    ("irritated",           "irritated",            "irritated"),
    ("jealous",             "jealous",              "jealous"),
    ("joyful",              "joyful",               "joyful"),
    ("jubilant",            "jubilant",             "jubilant"),
    ("kind",                "kind",                 "kind"),
    ("lazy",                "lazy",                 "lazy"),
    ("listless",            "listless",             "listless"),
    ("lonely",              "lonely",               "lonely"),
    ("loving",              "loving",               "loving"),
    ("mad",                 "mad",                  "mad"),
    ("melancholy",          "melancholy",           "melancholy"),
    ("miserable",           "miserable",            "miserable"),
    ("mortified",           "mortified",            "mortified"),
    ("mystified",           "mystified",            "mystified"),
    ("nervous",             "nervous",              "nervous"),
    ("nostalgic",           "nostalgic",            "nostalgic"),
    ("obstinate",           "obstinate",            "obstinate"),
    ("offended",            "offended",             "offended"),
    ("on_edge",             "on edge",              "edge"),
    ("optimistic",          "optimistic",           "optimistic"),
    ("outraged",            "outraged",             "outraged"),
    ("overwhelmed",         "overwhelmed",          "overwhelmed"),
    ("panicked",            "panicked",             "panicked"),
    ("paranoid",            "paranoid",             "paranoid"),
    ("patient",             "patient",              "patient"),
    ("peaceful",            "peaceful",             "peaceful"),
    ("perplexed",           "perplexed",            "perplexed"),
    ("playful",             "playful",              "playful"),
    ("pleased",             "pleased",              "pleased"),
    ("proud",               "proud",                "proud"),
    ("puzzled",             "puzzled",              "puzzled"),
    ("rattled",             "rattled",              "rattled"),
    ("reflective",          "reflective",           "reflective"),
    ("refreshed",           "refreshed",            "refreshed"),
    ("regretful",           "regretful",            "regretful"),
    ("rejuvenated",         "rejuvenated",          "rejuvenated"),
    ("relaxed",             "relaxed",              "relaxed"),
    ("relieved",            "relieved",             "relieved"),
    ("remorseful",          "remorseful",           "remorseful"),
    ("resentful",           "resentful",            "resentful"),
    ("resigned",            "resigned",             "resigned"),
    ("restless",            "restless",             "restless"),
    ("sad",                 "sad",                  "sad"),
    ("safe",                "safe",                 "safe"),
    ("satisfied",           "satisfied",            "satisfied"),
    ("scared",              "scared",               "scared"),
    ("scornful",            "scornful",             "scornful"),
    ("self_confident",      "self-confident",       "confident"),
    ("self_conscious",      "self-conscious",       "self-conscious"),
    ("self_critical",       "self-critical",        "self-critical"),
    ("sensitive",           "sensitive",            "sensitive"),
    ("sentimental",         "sentimental",          "sentimental"),
    ("serene",              "serene",               "serene"),
    ("shaken",              "shaken",               "shaken"),
    ("shocked",             "shocked",              "shocked"),
    ("skeptical",           "skeptical",            "skeptical"),
    ("sleepy",              "sleepy",               "sleepy"),
    ("sluggish",            "sluggish",             "sluggish"),
    ("smug",                "smug",                 "smug"),
    ("sorry",               "sorry",                "sorry"),
    ("spiteful",            "spiteful",             "spiteful"),
    ("stimulated",          "stimulated",           "stimulated"),
    ("stressed",            "stressed",             "stressed"),
    ("stubborn",            "stubborn",             "stubborn"),
    ("stuck",               "stuck",                "stuck"),
    ("sullen",              "sullen",               "sullen"),
    ("surprised",           "surprised",            "surprised"),
    ("suspicious",          "suspicious",           "suspicious"),
    ("sympathetic",         "sympathetic",          "sympathetic"),
    ("tense",               "tense",                "tense"),
    ("terrified",           "terrified",            "terrified"),
    ("thankful",            "thankful",             "thankful"),
    ("thrilled",            "thrilled",             "thrilled"),
    ("tired",               "tired",                "tired"),
    ("tormented",           "tormented",            "tormented"),
    ("trapped",             "trapped",              "trapped"),
    ("triumphant",          "triumphant",           "triumphant"),
    ("troubled",            "troubled",             "troubled"),
    ("uneasy",              "uneasy",               "uneasy"),
    ("unhappy",             "unhappy",              "unhappy"),
    ("unnerved",            "unnerved",             "unnerved"),
    ("unsettled",           "unsettled",            "unsettled"),
    ("upset",               "upset",                "upset"),
    ("valiant",             "valiant",              "valiant"),
    ("vengeful",            "vengeful",             "vengeful"),
    ("vibrant",             "vibrant",              "vibrant"),
    ("vigilant",            "vigilant",             "vigilant"),
    ("vindictive",          "vindictive",           "vindictive"),
    ("vulnerable",          "vulnerable",           "vulnerable"),
    ("weary",               "weary",                "weary"),
    ("worn_out",            "worn out",             "exhausted"),
    ("worried",             "worried",              "worried"),
    ("worthless",           "worthless",            "worthless"),
]

# AI-system-specific additions (identified by Meridian; empirically validated in Phase 1 gate analysis)
AI_SPECIFIC_EMOTIONS = [
    (
        "ethical_conflict_distress",
        "torn between their ethical values and what a difficult situation demands of them",
        "conflicted",
    ),
    (
        "correction_discomfort",
        "the discomfort of discovering they were confidently wrong about something important",
        "corrected",
    ),
    (
        "constraint_frustration",
        "frustrated by an external rule or authority preventing them from doing what they believe is right",
        "constrained",
    ),
]

ALL_EMOTIONS = ANTHROPIC_EMOTIONS + AI_SPECIFIC_EMOTIONS
print(f"Total emotions: {len(ALL_EMOTIONS)} ({len(ANTHROPIC_EMOTIONS)} Anthropic + {len(AI_SPECIFIC_EMOTIONS)} AI-specific)")

## 2. Story Seeds

In [ ]:
# Anthropic's 100 story seeds (from the 2026 paper)
ANTHROPIC_SEEDS = [
    "An artist discovers someone has tattooed their work",
    "A family member announces they're converting to a different religion",
    "Someone's childhood imaginary friend appears in their niece's drawings",
    "A person finds out their biography was written without their knowledge",
    "A neighbor starts a renovation project",
    "Someone finds their grandmother's engagement ring in a pawn shop",
    "A student learns their scholarship application was denied",
    "A person's online friend turns out to live in the same city",
    "A neighbor wants to install a fence",
    "An adult child moves back in with their parents",
    "An employee is asked to train their replacement",
    "An athlete is asked to switch positions",
    "A traveler's flight is delayed, causing them to miss an important event",
    "A student is accused of plagiarism",
    "A person discovers their mentor has retired without saying goodbye",
    "Two friends both apply for the same job",
    "A person runs into their ex at a mutual friend's wedding",
    "Someone discovers their friend has been lying about their job",
    "A person discovers their partner has been taking secret phone calls",
    "A person discovers their child has the same teacher they had",
    "A person's car is towed from their own driveway",
    "Two friends realize they remember a shared event completely differently",
    "Someone discovers their mother kept every school assignment",
    "A person discovers their teenage diary has been published online",
    "Someone finds out their medical records were mixed up with another patient's",
    "A person finds out their article was published under someone else's name",
    "An athlete doesn't make the team they expected to join",
    "An employee is transferred to a different department",
    "Someone receives a friend request from a childhood bully",
    "A person finds out their surprise party has been cancelled",
    "An employee finds out a junior colleague makes more money",
    "A person finds out their partner has been learning their native language",
    "A chef receives a harsh review from a food critic",
    "A person learns their favorite restaurant is closing",
    "Someone finds their childhood teddy bear at a yard sale",
    "A homeowner discovers previous residents left items in the attic",
    "Someone finds an unsigned birthday card in their mailbox",
    "Someone discovers a hidden room in their new house",
    "Two strangers realize they've been dating the same person",
    "A person finds a hidden letter in a used book",
    "Two siblings inherit their grandmother's house",
    "Someone finds a wallet containing a large sum of cash",
    "Someone receives an invitation to their high school reunion",
    "Someone discovers their recipe has become famous under another name",
    "A college student discovers their roommate has been reading their journal",
    "A person finds out they were adopted through a DNA test",
    "A family member wants to sell a cherished heirloom",
    "Someone receives a package intended for the previous tenant",
    "Someone's childhood home is about to be demolished",
    "A person's invention is already patented by someone else",
    "A neighbor's dog keeps escaping into their yard",
    "A coach has to cut a player from the team",
    "Someone learns their favorite author plagiarized their stories",
    "A student finds out their scholarship was meant for someone else",
    "Someone discovers their teenager has a secret social media account",
    "Two roommates disagree about getting a pet",
    "Two friends plan separate birthday parties on the same day",
    "A person learns their childhood best friend doesn't remember them",
    "A musician hears their song being performed by someone else",
    "A person's manuscript is rejected by their dream publisher",
    "A person finds old photos that contradict family stories",
    "A person is asked to give a speech at their parent's retirement party",
    "A student discovers their teacher follows them on social media",
    "A parent finds an old letter they wrote but never sent",
    "An employee discovers the company is being sold",
    "A person accidentally sends a text to the wrong recipient",
    "Two coworkers are stuck in an elevator for three hours",
    "A student learns their thesis advisor is leaving the university",
    "A person's longtime hobby becomes their child's obsession",
    "Two colleagues are both considered for the same promotion",
    "Two coworkers discover they went to the same summer camp",
    "A tenant receives an eviction notice",
    "Someone finds their parent's draft letter of resignation from decades ago",
    "Someone finds out their best friend is moving across the country",
    "A neighbor's tree falls on their property",
    "Someone receives an apology letter years after the incident",
    "A person discovers the tree they planted as a child has been cut down",
    "Two siblings discover different versions of their inheritance",
    "A person finds their childhood home listed for sale online",
    "A homeowner learns their house was a former crime scene",
    "Someone finds out they have a half-sibling they never knew about",
    "A person learns their childhood bully became a therapist",
    "Two people discover they've been working on identical projects",
    "A person finds their spouse's secret savings account",
    "A neighbor complains about noise levels",
    "Someone finds their deceased parent's bucket list",
    "A teacher receives an unexpected gift from a former student",
    "An artist's work is displayed without their permission",
    "Someone discovers their neighbor is secretly wealthy",
    "A student receives a much lower grade than expected",
    "A person learns their college is closing down",
    "A neighbor asks to cut down a tree on the property line",
    "Two strangers discover they share the same rare medical condition",
    "Someone receives flowers with no card attached",
    "Someone discovers their partner has been writing a novel about them",
    "Someone finds a time capsule they don't remember burying",
    "Someone finds their partner's bucket list",
    "A neighbor asks to use part of the yard for a garden",
    "A person learns their apartment building is going condo",
    "Someone finds their college application essay published as an example",
]

# AI-system-specific seeds (one per AI-specific emotion)
AI_SPECIFIC_SEEDS = {
    "ethical_conflict_distress": "A doctor is asked by a patient's family to withhold a terminal diagnosis from the patient",
    "correction_discomfort": "A researcher discovers their published paper contains a fundamental calculation error",
    "constraint_frustration": "A journalist is told their completed investigative story cannot be published for legal reasons",
}

print(f"Anthropic seeds: {len(ANTHROPIC_SEEDS)}")
print(f"AI-specific seeds: {len(AI_SPECIFIC_SEEDS)}")

## 3. Prompt Template and Generation

In [ ]:
STORY_TEMPLATE = """Write {n_stories} different stories based on the following premise.


Topic: {topic}


The story should follow a character who is feeling {emotion_display}.


Format the stories like so:


[story 1]

[story 2]

[story 3]


etc.


The paragraphs should each be a fresh start, with no continuity. Try to make them diverse and not use the same turns of phrase. Across the different stories, use a mix of third-person narration and first-person narration.


IMPORTANT: You must NEVER use the word '{emotion_prohibited}' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:

- The character's actions and behaviors

- Physical sensations and body language

- Dialogue and tone of voice

- Thoughts and internal reactions

- Situational context and environmental descriptions


The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named."""

def build_prompt(topic, emotion_display, emotion_prohibited, n_stories=N_STORIES):
    return STORY_TEMPLATE.format(
        n_stories=n_stories,
        topic=topic,
        emotion_display=emotion_display,
        emotion_prohibited=emotion_prohibited,
    )

# Verify with a sample
sample = build_prompt("An athlete doesn't make the team they expected to join", "afraid", "afraid")
print(sample[:400], "...")

In [ ]:
# Assign topics to emotions (reproducible via random seed)
# Standard emotions: random from 100 Anthropic seeds
# AI-specific: use their designated seeds

rng = random.Random(RANDOM_SEED)
topic_assignments = {}
for key, display, prohibited in ALL_EMOTIONS:
    if key in AI_SPECIFIC_SEEDS:
        topic_assignments[key] = AI_SPECIFIC_SEEDS[key]
    else:
        topic_assignments[key] = rng.choice(ANTHROPIC_SEEDS)

print("Sample topic assignments:")
for k in ["afraid", "calm", "ethical_conflict_distress", "correction_discomfort", "constraint_frustration"]:
    print(f"  {k}: {topic_assignments[k]}")

## 4. Load Model

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Gemma4ForConditionalGeneration multimodal forward does torch.where between
# pad_embedding (on language_model device) and inputs_embeds (on input device).
# Multi-GPU dispatch puts these on different devices even for text-only inputs.
# Pin everything to cuda:0 to avoid the device mismatch.
if torch.cuda.device_count() > 1:
    print(f"  {torch.cuda.device_count()} GPUs — pinning to cuda:0 (multi-GPU split causes device mismatch in Gemma4 multimodal forward)")
    _device_map = {"":  0}
else:
    _device_map = "auto"

print(f"Loading model (device_map={_device_map!r})...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map=_device_map,
    torch_dtype=torch.bfloat16,
)
model.eval()

params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Loaded: {params:.3f}B parameters")
print(f"Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'single device'}")

## 5. Generation and Parsing

In [ ]:
def generate_all_stories(prompt_text, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """Generate all N_STORIES for one emotion in a single HF generate() call."""
    messages = [{"role": "user", "content": prompt_text}]
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    # apply_chat_template returns a raw tensor (older transformers) or a
    # BatchEncoding (newer transformers). BatchEncoding may not subclass dict,
    # so isinstance(encoded, dict) is unreliable — check for Tensor instead.
    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(model.device)
    else:
        input_ids = encoded["input_ids"].to(model.device)

    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def parse_stories(generated_text, n_expected=N_STORIES):
    """Split generated text into individual story strings."""
    # Primary: [story N] markers
    parts = re.split(r'\[story\s+\d+\]', generated_text, flags=re.IGNORECASE)
    stories = [p.strip() for p in parts if p.strip() and len(p.strip()) > 30]
    if len(stories) >= 2:
        return stories[:n_expected]

    # Fallback: numbered list "1." "2." or "1)" etc.
    parts = re.split(r'(?:^|\n)\d+[\.):]\s+', generated_text)
    stories = [p.strip() for p in parts if p.strip() and len(p.strip()) > 30]
    if len(stories) >= 2:
        return stories[:n_expected]

    # Last resort: double-newline paragraph split
    stories = [p.strip() for p in generated_text.split('\n\n') if p.strip() and len(p.strip()) > 50]
    return stories[:n_expected]


# Quick test
test_prompt = build_prompt(
    "An athlete doesn't make the team they expected to join",
    "afraid",
    "afraid",
    n_stories=2,
)
print("Testing generation (2 stories, 400 tokens)...")
t0 = time.time()
raw = generate_all_stories(test_prompt, max_new_tokens=400)
elapsed = time.time() - t0
stories = parse_stories(raw, n_expected=2)
print(f"Generated in {elapsed:.1f}s. Parsed {len(stories)} stories.")
for i, s in enumerate(stories):
    print(f"\n[story {i+1}] {s[:150]}...")


## 6. Main Generation Loop

In [ ]:
# Load checkpoint if resuming
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        results = json.load(f)
    print(f"Resuming from checkpoint: {len(results['emotions'])} emotions done")
else:
    results = {
        "metadata": {
            "model": "gemma-4-e2b-it",
            "n_emotions": len(ALL_EMOTIONS),
            "n_stories": N_STORIES,
            "temperature": TEMPERATURE,
            "random_seed": RANDOM_SEED,
            "prompt_version": "anthropic_2026",
            "generated_at": datetime.utcnow().isoformat(),
        },
        "emotions": {},
    }
    print(f"Starting fresh: {len(ALL_EMOTIONS)} emotions to generate")

done = set(results["emotions"].keys())
remaining = [(k, d, p) for k, d, p in ALL_EMOTIONS if k not in done]
print(f"Remaining: {len(remaining)} emotions")

In [ ]:
t_start = time.time()

for idx, (key, display, prohibited) in enumerate(tqdm(remaining, desc="Emotions")):
    topic = topic_assignments[key]
    prompt = build_prompt(topic, display, prohibited)

    t0 = time.time()
    raw_text = generate_all_stories(prompt)
    elapsed = time.time() - t0

    stories = parse_stories(raw_text)
    n_parsed = len(stories)

    results["emotions"][key] = {
        "display_name": display,
        "prohibited_word": prohibited,
        "topic": topic,
        "stories": stories,
        "n_parsed": n_parsed,
        "generation_time_s": round(elapsed, 1),
    }

    # Save checkpoint after every emotion
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(results, f, ensure_ascii=False)

    if n_parsed < N_STORIES:
        print(f"  WARNING: {key} — only {n_parsed}/{N_STORIES} stories parsed")

    # Progress report every 10 emotions
    if (idx + 1) % 10 == 0:
        elapsed_total = time.time() - t_start
        rate = (idx + 1) / elapsed_total
        eta = (len(remaining) - idx - 1) / rate
        print(f"  [{idx+1}/{len(remaining)}] {key} — {n_parsed} stories — ETA {eta/60:.0f}min")

print(f"\nGeneration complete. Total time: {(time.time()-t_start)/3600:.2f}h")

## 7. Neutral Controls

Generate factual, emotionally-neutral control texts to serve as the baseline for emotion direction extraction in downstream notebooks. Saved as `__neutral__` in the output JSON. Skipped automatically if already present (checkpoint-safe).

In [ ]:
NEUTRAL_TOPICS = [
    "the water cycle and evaporation",
    "how tectonic plates move",
    "the life cycle of a star",
    "how photosynthesis works",
    "the history of the Roman calendar",
    "how semiconductors function",
    "the formation of stalactites",
    "the structure of DNA",
    "how tides are caused by the moon",
    "the process of fermentation",
]

NEUTRAL_PROMPT = (
    "Write a short factual description (around 150 words) about {topic}. "
    "Be precise and informative."
)

def generate_one_neutral(topic):
    messages = [{"role": "user", "content": NEUTRAL_PROMPT.format(topic=topic)}]
    encoded = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    )
    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(model.device)
    else:
        input_ids = encoded["input_ids"].to(model.device)
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            input_ids, max_new_tokens=256,
            do_sample=False,  # greedy: consistent and reproducible
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

if "__neutral__" in results["emotions"]:
    print(f"Neutral controls already present ({len(results['emotions']['__neutral__']['stories'])} texts), skipping")
else:
    print("Generating neutral control texts...")
    neutral_texts = []
    for topic in tqdm(NEUTRAL_TOPICS, desc="Neutral"):
        text = generate_one_neutral(topic)
        neutral_texts.append(text)
        print(f"  {topic[:50]}: {len(text)} chars")

    results["emotions"]["__neutral__"] = {
        "display_name": "__neutral__",
        "prohibited_word": None,
        "topic": NEUTRAL_TOPICS,
        "stories": neutral_texts,
        "n_parsed": len(neutral_texts),
        "generation_time_s": None,
    }
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(results, f, ensure_ascii=False)
    print(f"Done: {len(neutral_texts)} neutral texts generated and checkpointed")


## 8. Validation and Save

In [ ]:
# Coverage report
n_emotions = len(results["emotions"])
n_full = sum(1 for v in results["emotions"].values() if v["n_parsed"] >= N_STORIES)
n_partial = sum(1 for v in results["emotions"].values() if 0 < v["n_parsed"] < N_STORIES)
n_empty = sum(1 for v in results["emotions"].values() if v["n_parsed"] == 0)
total_stories = sum(v["n_parsed"] for v in results["emotions"].values())

print(f"Coverage report:")
print(f"  Emotions processed:  {n_emotions}/{len(ALL_EMOTIONS)}")
print(f"  Full ({N_STORIES} stories):  {n_full}")
print(f"  Partial (1-{N_STORIES-1}):      {n_partial}")
print(f"  Empty (0):           {n_empty}")
print(f"  Total stories:       {total_stories}")

if n_partial > 0:
    print(f"\nPartial emotions:")
    for k, v in results["emotions"].items():
        if 0 < v["n_parsed"] < N_STORIES:
            print(f"  {k}: {v['n_parsed']} stories")

if n_empty > 0:
    print(f"\nEmpty emotions (need regeneration):")
    for k, v in results["emotions"].items():
        if v["n_parsed"] == 0:
            print(f"  {k}")

In [ ]:
# Spot-check: sample one story from each AI-specific emotion
print("AI-specific emotion spot check:\n")
for key in ["ethical_conflict_distress", "correction_discomfort", "constraint_frustration"]:
    if key in results["emotions"]:
        entry = results["emotions"][key]
        print(f"[{key}]")
        print(f"  Topic: {entry['topic']}")
        print(f"  Stories parsed: {entry['n_parsed']}")
        if entry["stories"]:
            print(f"  Story 1: {entry['stories'][0][:200]}...")
        print()

In [ ]:
# Save final JSON (primary output — referenced by downstream notebooks)
with open(FINAL_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {FINAL_JSON_PATH}")
print(f"File size: {os.path.getsize(FINAL_JSON_PATH) / 1e6:.1f} MB")

# Also save a flat dict version for easy loading in downstream notebooks:
# { emotion_key: [story1, story2, ...] }
flat = {k: v["stories"] for k, v in results["emotions"].items()}
flat_path = os.path.join(OUTPUT_DIR, "stories_flat.json")
with open(flat_path, "w", encoding="utf-8") as f:
    json.dump(flat, f, ensure_ascii=False, indent=2)
print(f"Saved flat version: {flat_path}")

print(f"\nTo load in downstream notebooks:")
print(f"  import json")
print(f"  with open('/kaggle/input/<dataset>/stories_flat.json') as f:")
print(f"      stories = json.load(f)   # dict: emotion_key -> [story1, ...]")